# Data Mining Project 10: Social Media Trend Analysis

## Project Overview

This project analyzes social media posts to identify **trending topics, public interests, and engagement patterns**. The Social Media Trend 2024 dataset is collected from Kaggle and processed using text cleaning, hashtag analysis, keyword analysis, engagement analysis, and visualization.

### Objectives
1. Collect and inspect the social media dataset.
2. Clean and preprocess tweet text.
3. Identify the most frequently discussed trending topics.
4. Identify common public-interest keywords.
5. Analyze engagement using likes and retweets.
6. Present the findings through clear analytical visualizations.


## 1. Import Required Libraries


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub


## 2. Collect and Load the Social Media Dataset

The dataset is downloaded using KaggleHub. The CSV file is located automatically so that the notebook does not depend on a hard-coded filename.


In [ ]:
path = kagglehub.dataset_download("muskantariq/social-media-trend-2024")
print("Dataset path:", path)
print("Files:", os.listdir(path))


In [ ]:
# Locate the CSV file automatically
csv_files = [f for f in os.listdir(path) if f.lower().endswith(".csv")]
print("CSV files found:", csv_files)

if not csv_files:
    raise FileNotFoundError("No CSV file was found in the downloaded dataset.")

df = pd.read_csv(os.path.join(path, csv_files[0]))
df.head()


## 3. Explore and Understand the Dataset


In [ ]:
print("Dataset Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())


In [ ]:
df.info()


In [ ]:
print("Descriptive Statistics:")
df.describe(include="all").T


## 4. Data Cleaning and Text Preprocessing

The column names are standardized, duplicate records are removed, records without tweet or hashtag information are excluded, and tweet text is cleaned by removing URLs, user mentions, special characters, and extra spaces. Date and engagement columns are also converted to suitable numeric/date formats where available.


In [ ]:
# Standardize column names
df.columns = (
    df.columns.str.strip().str.lower().str.replace(" ", "_")
)

print("Standardized columns:")
print(df.columns.tolist())


In [ ]:
# Remove duplicate rows and records without required text fields
df = df.drop_duplicates()
df = df.dropna(subset=["tweet", "hashtag"])

# Convert text fields to strings
df["tweet"] = df["tweet"].astype(str)
df["hashtag"] = df["hashtag"].astype(str)

# Clean tweet text
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"[^a-zA-Z0-9#\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_tweet"] = df["tweet"].apply(clean_text)

# Convert date and engagement columns when present
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

for col in ["likes", "retweets"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

print("Shape after preprocessing:", df.shape)
df[["tweet", "clean_tweet", "hashtag"]].head()


## 5. Identify Trending Topics

Hashtag frequency is used to identify the topics that appear most frequently in the dataset.


In [ ]:
hashtags = (
    df["hashtag"].str.lower().str.replace("#", "", regex=False)
    .str.split().explode().str.strip()
)

hashtags = hashtags[hashtags != ""]
trending_topics = hashtags.value_counts().head(10)

print("Top 10 Trending Topics:")
print(trending_topics)


## 6. Identify Public Interests

Common words from cleaned tweets are extracted after removing frequently occurring stop words. The most frequent remaining keywords represent major public interests in the dataset.


In [ ]:
stop_words = {
    "the", "and", "for", "that", "this", "with", "from", "are",
    "was", "were", "have", "has", "had", "you", "your", "our",
    "they", "their", "about", "what", "when", "where", "which",
    "will", "would", "could", "should", "into", "just", "more",
    "than", "then", "them", "there", "here", "its", "it's",
    "but", "not", "all", "can", "get", "got", "out", "who",
    "how", "why", "too", "very", "his", "her", "she", "him",
    "been", "being", "also"
}

words = (
    df["clean_tweet"].str.replace("#", " ", regex=False)
    .str.split().explode()
)

words = words[words.str.len().gt(2) & ~words.isin(stop_words)]
public_interests = words.value_counts().head(15)

print("Most Discussed Public Interests / Keywords:")
print(public_interests)


## 7. Analyze Social Media Engagement

Engagement is calculated as the sum of likes and retweets. This helps identify posts and hashtags that receive stronger audience interaction.


In [ ]:
df["engagement"] = df["likes"] + df["retweets"]

top_engaging_posts = (
    df.sort_values("engagement", ascending=False)
    [["tweet", "hashtag", "likes", "retweets", "engagement"]]
    .head(10)
)

print("Top 10 Most Engaging Posts:")
top_engaging_posts


In [ ]:
hashtag_engagement = (
    df.groupby("hashtag")["engagement"]
    .mean().sort_values(ascending=False).head(10)
)

print("Top Hashtags by Average Engagement:")
print(hashtag_engagement)


## 8. Analytical Visualizations


In [ ]:
plt.figure(figsize=(10, 5))
trending_topics.sort_values().plot(kind="barh")
plt.title("Top 10 Trending Topics")
plt.xlabel("Number of Posts")
plt.ylabel("Hashtag")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
public_interests.sort_values().plot(kind="barh")
plt.title("Top Public Interest Keywords")
plt.xlabel("Number of Mentions")
plt.ylabel("Keyword")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
hashtag_engagement.sort_values().plot(kind="barh")
plt.title("Top Hashtags by Average Engagement")
plt.xlabel("Average Engagement")
plt.ylabel("Hashtag")
plt.tight_layout()
plt.show()


In [ ]:
if "date" in df.columns:
    daily_posts = (
        df.dropna(subset=["date"])
        .groupby(df["date"].dt.date).size()
    )

    plt.figure(figsize=(10, 5))
    daily_posts.plot()
    plt.title("Social Media Posting Trend Over Time")
    plt.xlabel("Date")
    plt.ylabel("Number of Posts")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## 9. Final Findings


In [ ]:
print("===== SOCIAL MEDIA TREND ANALYSIS SUMMARY =====")

print("\nTop Trending Topics:")
print(trending_topics.head(5))

print("\nTop Public Interest Keywords:")
print(public_interests.head(5))

print("\nHighest Average Engagement Hashtags:")
print(hashtag_engagement.head(5))

print("\nTotal Posts Analyzed:", len(df))
print("Average Likes:", round(df["likes"].mean(), 2))
print("Average Retweets:", round(df["retweets"].mean(), 2))
print("Average Engagement:", round(df["engagement"].mean(), 2))


# Conclusion

The Social Media Trend 2024 dataset was successfully collected, cleaned, and analyzed. Duplicate records and incomplete tweet/hashtag entries were removed, while tweet text was standardized by eliminating URLs, mentions, special characters, and unnecessary spaces.

Hashtag frequency analysis was used to identify trending topics, and keyword frequency analysis was used to understand major public interests. Engagement was calculated from likes and retweets to identify highly interactive posts and hashtags.

The visualizations provide a clear representation of trending topics, public interests, engagement patterns, and posting activity. Overall, the project demonstrates how data mining and text-based analysis can be used to extract useful insights from social media data and support data-driven social media and marketing decisions.